In [1]:
def wrap_file(f: int) -> int:
    return ((f - 1) % 8) + 1

def cell(file_no: int, letter: str) -> str:
    return f"{file_no}{letter}"

def expand_templates(templates):
    pairs = []
    for src_letter, df, dst_letter in templates:
        for f in range(1, 9):
            src = cell(f, src_letter)
            dst = cell(wrap_file(f + df), dst_letter)
            pairs.append((src, dst))
    return pairs

def build_links_from_templates(link_templates):
    links = {}
    for name, templates in link_templates.items():
        links[name] = expand_templates(templates)
    return links

In [2]:
def expand_center_templates(n_to_center, center_to_g):
    pairs = []

    # ?G -> OO
    for letter, dst in n_to_center:
        for f in range(1, 9):
            src = f"{f}{letter}"
            pairs.append((src, "00"))

    # OO -> ?G
    for src, letter in center_to_g:
        for f in range(1, 9):
            dst = f"{f}{letter}"
            pairs.append(("00", dst))

    return pairs

In [3]:
N_TO_CENTER = [
    ("G", "00"),
]
CENTER_LINKS = {
    "1s": [("00", "1G")],
    "2s": [("00", "2G")],
    "3s": [("00", "3G")],
    "4s": [("00", "4G")],
    "5s": [("00", "5G")],
    "6s": [("00", "6G")],
    "7s": [("00", "7G")],
    "8s": [("00", "8G")],
}

In [4]:
def build_all_links():
    links = {}

    # 기존 템플릿 기반 링크
    base_links = build_links_from_templates(LINK_TEMPLATES)
    links.update(base_links)

    # n에서 OO로 올라가는 추가 링크
    links["n"] += expand_center_templates(
        N_TO_CENTER,
        []
    )

    links=(links|CENTER_LINKS)
    return links

In [5]:
def walk(adj, start, link_name, occupied_cells=None):
    return follow_move_sequence(
        adj, start, [link_name],
        occupied_cells=occupied_cells,
        require_empty_intermediate=False
    )

def walk2(adj, start, a, b, occupied_cells=None):
    return follow_move_sequence(
        adj, start, [a, b],
        occupied_cells=occupied_cells,
        require_empty_intermediate=True
    )

def walk3(adj, start, a, b, c, occupied_cells=None):
    return follow_move_sequence(
        adj, start, [a, b, c],
        occupied_cells=occupied_cells,
        require_empty_intermediate=True
    )

def walk4(adj, start, a, b, c, d, occupied_cells=None):
    return follow_move_sequence(
        adj, start, [a, b, c, d],
        occupied_cells=occupied_cells,
        require_empty_intermediate=True
    )

def fly2(adj, start, a, b, occupied_cells=None):
    return follow_move_sequence(
        adj, start, [a, b],
        occupied_cells=occupied_cells,
        require_empty_intermediate=False
    )

def fly3(adj, start, a, b, c, occupied_cells=None):
    return follow_move_sequence(
        adj, start, [a, b, c],
        occupied_cells=occupied_cells,
        require_empty_intermediate=False
    )

def fly4(adj, start, a, b, c, d, occupied_cells=None):
    return follow_move_sequence(
        adj, start, [a, b, c, d],
        occupied_cells=occupied_cells,
        require_empty_intermediate=False
    )

In [6]:
def can_piece_move(adj, start, dest, move_def, occupied_cells=None):
    """
    move_def 예:
      ("walk", "n")
      ("walk2", "e", "e")
      ("fly3", "ne", "e", "s")
    """
    op = move_def[0]
    args = move_def[1:]

    if op == "walk":
        targets = walk(adj, start, args[0], occupied_cells)
    elif op == "walk2":
        targets = walk2(adj, start, args[0], args[1], occupied_cells)
    elif op == "walk3":
        targets = walk3(adj, start, args[0], args[1], args[2], occupied_cells)
    elif op == "walk4":
        targets = walk4(adj, start, args[0], args[1], args[2], args[3], occupied_cells)
    elif op == "fly2":
        targets = fly2(adj, start, args[0], args[1], occupied_cells)
    elif op == "fly3":
        targets = fly3(adj, start, args[0], args[1], args[2], occupied_cells)
    elif op == "fly4":
        targets = fly4(adj, start, args[0], args[1], args[2], args[3], occupied_cells)
    else:
        raise ValueError(f"unknown move operator: {op}")

    return dest in targets

In [7]:
from collections import defaultdict

# -----------------------------
# 1. 링크 템플릿
# -----------------------------
LINK_TEMPLATES = {
    "n": [
        ("K", 0, "J"),
        ("J", 0, "I"),
        ("B", 0, "A"),
        ("C", 0, "D"),
        ("D", 0, "E"),
        ("I", 0, "H"),
        ("H", 0, "G"),
        ("E", 0, "F"),
        ("F", 0, "G"),
    ],

    "en": [
        ("A", 0, "I"),
    ],

    "wn": [
        ("A", 0, "E"),
    ],

    "e": [
        ("G", -1, "G"),
        ("F", 0, "H"),
        ("H", -1, "F"),
        ("E", 0, "I"),
        ("I", -1, "E"),
        ("D", 0, "A"),
        ("A", 0, "J"),
        ("J", -1, "D"),
        ("C", 0, "B"),
        ("B", 0, "K"),
        ("K", -1, "C"),
    ],

    "w": [
        ("G", +1, "G"),
        ("H", 0, "F"),
        ("F", +1, "H"),
        ("I", 0, "E"),
        ("E", +1, "I"),
        ("J", 0, "A"),
        ("A", 0, "D"),
        ("D", +1, "J"),
        ("K", 0, "B"),
        ("B", 0, "C"),
        ("C", +1, "K"),
    ],

    "s": [
        ("I", 0, "J"),
        ("J", 0, "K"),
        ("A", 0, "B"),
        ("I", 0, "A"),
        ("E", 0, "D"),
        ("D", 0, "C"),
        ("H", 0, "I"),
        ("F", 0, "E"),
    ],

    "ws": [
        ("I", 0, "A"),
        ("G", 0, "F"),
    ],

    "es": [
        ("E", 0, "A"),
        ("G", 0, "H"),
    ],

    "ne": [
        ("B", 0, "J"),
        ("J", -1, "E"),
        ("E", 0, "H"),
        ("H", -1, "G"),
        ("C", 0, "A"),
        ("I", -1, "F"),
        ("K", -1, "D"),
    ],

    "nw": [
        ("B", 0, "D"),
        ("D", +1, "I"),
        ("I", 0, "F"),
        ("F", +1, "G"),
        ("K", 0, "A"),
        ("E", +1, "H"),
        ("C", +1, "J"),
    ],

    "se": [
        ("D", 0, "B"),
        ("I", -1, "D"),
        ("F", 0, "I"),
        ("G", -1, "F"),
        ("A", 0, "K"),
        ("H", -1, "E"),
        ("J", -1, "C"),
    ],

    "sw": [
        ("J", 0, "B"),
        ("E", +1, "J"),
        ("H", 0, "E"),
        ("G", +1, "H"),
        ("A", 0, "C"),
        ("F", +1, "I"),
        ("D", +1, "K"),
    ],

    "s3": [
        ("G", 0, "A"),
    ],
    "n3": [
        ("A", 0, "G"),
    ],
}


# -----------------------------
# 2. 중심 링크 추가
#   - n : ?G -> OO
#   - 1s~8s : OO -> ?G
# -----------------------------
def wrap_file(f: int) -> int:
    return ((f - 1) % 8) + 1

def cell(file_no: int, letter: str) -> str:
    if letter == "OO":
        return "00"
    return f"{file_no}{letter}"

def expand_templates(templates):
    pairs = []
    for src_letter, df, dst_letter in templates:
        for f in range(1, 9):
            src = cell(f, src_letter)
            dst = cell(wrap_file(f + df), dst_letter)
            pairs.append((src, dst))
    return pairs

def build_links_from_templates(link_templates):
    links = {}
    for name, templates in link_templates.items():
        links[name] = expand_templates(templates)
    return links

def add_center_links_split(links):
    # n: ?G -> OO
    links.setdefault("n", [])
    for f in range(1, 9):
        links["n"].append((f"{f}G", "00"))

    # 1s ~ 8s: OO -> ?G
    for f in range(1, 9):
        links[f"{f}s"] = [("00", f"{f}G")]

    return links

def build_adjacency(links):
    adj = {}
    for name, pairs in links.items():
        d = defaultdict(set)
        for src, dst in pairs:
            d[src].add(dst)
        adj[name] = dict(d)
    return adj


# -----------------------------
# 3. 기물 행마 정의
#   지금은 pawn만 넣음
# -----------------------------
PIECES = {
    "pawn": [
        ("walk", "n"),
        ("walk", "s"),
        ("walk", "e"),
        ("walk", "w"),
        ("walk", "en"),
        ("walk", "wn"),
        ("walk", "es"),
        ("walk", "ws"),
        ("walk", "1s"),
        ("walk", "2s"),
        ("walk", "3s"),
        ("walk", "4s"),
        ("walk", "5s"),
        ("walk", "6s"),
        ("walk", "7s"),
        ("walk", "8s"),
    ],

    "raven": [
        ("walk", "n"),
        ("walk", "s"),
        ("walk", "e"),
        ("walk", "w"),
        ("walk", "en"),
        ("walk", "wn"),
        ("walk", "es"),
        ("walk", "ws"),
        ("walk", "1s"),
        ("walk", "2s"),
        ("walk", "3s"),
        ("walk", "4s"),
        ("walk", "5s"),
        ("walk", "6s"),
        ("walk", "7s"),
        ("walk", "8s"),
        ("walk", "ne"),
        ("walk", "se"),
        ("walk", "nw"),
        ("walk", "sw"),
    ],

    "horse": [
        ("walk2", "n", "e"),
        ("walk2", "s", "e"),
        ("walk2", "n", "w"),
        ("walk2", "s", "w"),
        ("walk2", "e", "e"),
        ("walk2", "w", "w"),
        ("walk2", "e", "n"),
        ("walk2", "e", "s"),
        ("walk2", "w", "n"),
        ("walk2", "w", "s"),
        ("walk2", "en", "e"),
        ("walk2", "wn", "e"),
        ("walk2", "es", "e"),
        ("walk2", "ws", "e"),
        ("walk2", "en", "w"),
        ("walk2", "wn", "w"),
        ("walk2", "es", "w"),
        ("walk2", "ws", "w"),
        ("walk2", "e", "en"),
        ("walk2", "e", "wn"),
        ("walk2", "e", "es"),
        ("walk2", "e", "ws"),
        ("walk2", "w", "en"),
        ("walk2", "w", "wn"),
        ("walk2", "w", "es"),
        ("walk2", "w", "ws"),
        ("walk2", "1s", "e"),
        ("walk2", "2s", "e"),
        ("walk2", "3s", "e"),
        ("walk2", "4s", "e"),
        ("walk2", "5s", "e"),
        ("walk2", "6s", "e"),
        ("walk2", "7s", "e"),
        ("walk2", "8s", "e"),
        ("walk2", "1s", "w"),
        ("walk2", "2s", "w"),
        ("walk2", "3s", "w"),
        ("walk2", "4s", "w"),
        ("walk2", "5s", "w"),
        ("walk2", "6s", "w"),
        ("walk2", "7s", "w"),
        ("walk2", "8s", "w"),
    ],

    "tiger": [
        ("walk3", "e", "e", "e"),
        ("walk3", "w", "w", "w"),
        ("walk3", "n", "e", "e"),
        ("walk3", "s", "e", "e"),
        ("walk3", "n", "w", "w"),
        ("walk3", "s", "w", "w"),
        ("walk3", "e", "n", "e"),
        ("walk3", "e", "s", "e"),
        ("walk3", "w", "n", "w"),
        ("walk3", "w", "s", "w"),
        ("walk3", "e", "n", "w"),
        ("walk3", "e", "s", "w"),
        ("walk3", "w", "n", "e"),
        ("walk3", "w", "s", "e"),
        ("walk3", "e", "e", "n"),
        ("walk3", "w", "w", "n"),
        ("walk3", "e", "e", "s"),
        ("walk3", "w", "w", "s"),
        ("walk3", "en", "e", "e"),
        ("walk3", "es", "e", "e"),
        ("walk3", "en", "w", "w"),
        ("walk3", "es", "w", "w"),
        ("walk3", "e", "en", "e"),
        ("walk3", "e", "es", "e"),
        ("walk3", "w", "en", "w"),
        ("walk3", "w", "es", "w"),
        ("walk3", "e", "en", "w"),
        ("walk3", "e", "es", "w"),
        ("walk3", "w", "en", "e"),
        ("walk3", "w", "es", "e"),
        ("walk3", "e", "e", "en"),
        ("walk3", "w", "w", "en"),
        ("walk3", "e", "e", "es"),
        ("walk3", "w", "w", "es"),
        ("walk3", "wn", "e", "e"),
        ("walk3", "ws", "e", "e"),
        ("walk3", "wn", "w", "w"),
        ("walk3", "ws", "w", "w"),
        ("walk3", "e", "wn", "e"),
        ("walk3", "e", "ws", "e"),
        ("walk3", "w", "wn", "w"),
        ("walk3", "w", "ws", "w"),
        ("walk3", "e", "wn", "w"),
        ("walk3", "e", "ws", "w"),
        ("walk3", "w", "wn", "e"),
        ("walk3", "w", "ws", "e"),
        ("walk3", "e", "e", "wn"),
        ("walk3", "w", "w", "wn"),
        ("walk3", "e", "e", "ws"),
        ("walk3", "w", "w", "ws"),
        ("walk3", "1s", "e", "e"),
        ("walk3", "2s", "e", "e"),
        ("walk3", "3s", "e", "e"),
        ("walk3", "4s", "e", "e"),
        ("walk3", "5s", "e", "e"),
        ("walk3", "6s", "e", "e"),
        ("walk3", "7s", "e", "e"),
        ("walk3", "8s", "e", "e"),
        ("walk3", "1s", "w", "w"),
        ("walk3", "2s", "w", "w"),
        ("walk3", "3s", "w", "w"),
        ("walk3", "4s", "w", "w"),
        ("walk3", "5s", "w", "w"),
        ("walk3", "6s", "w", "w"),
        ("walk3", "7s", "w", "w"),
        ("walk3", "8s", "w", "w"),
    ],

    "elephant": [
        ("walk4", "e", "e", "e", "e"),
        ("walk4", "w", "w", "w", "w"),
        ("walk4", "n", "e", "e", "e"),
        ("walk4", "s", "e", "e", "e"),
        ("walk4", "n", "w", "w", "w"),
        ("walk4", "s", "w", "w", "w"),
        ("walk4", "e", "n", "e", "e"),
        ("walk4", "e", "s", "e", "e"),
        ("walk4", "w", "n", "w", "w"),
        ("walk4", "w", "s", "w", "w"),
        ("walk4", "e", "e", "n", "e"),
        ("walk4", "e", "e", "s", "e"),
        ("walk4", "w", "w", "n", "w"),
        ("walk4", "w", "w", "s", "w"),
        ("walk4", "e", "e", "e", "n"),
        ("walk4", "e", "e", "e", "s"),
        ("walk4", "w", "w", "w", "n"),
        ("walk4", "w", "w", "w", "s"),
        ("walk4", "e", "n", "w", "w"),
        ("walk4", "e", "s", "w", "w"),
        ("walk4", "w", "n", "e", "e"),
        ("walk4", "w", "s", "e", "e"),
        ("walk4", "e", "e", "n", "w"),
        ("walk4", "e", "e", "s", "w"),
        ("walk4", "w", "w", "n", "e"),
        ("walk4", "w", "w", "s", "e"),
        ("walk4", "en", "e", "e", "e"),
        ("walk4", "es", "e", "e", "e"),
        ("walk4", "en", "w", "w", "w"),
        ("walk4", "es", "w", "w", "w"),
        ("walk4", "e", "en", "e", "e"),
        ("walk4", "e", "es", "e", "e"),
        ("walk4", "w", "en", "w", "w"),
        ("walk4", "w", "es", "w", "w"),
        ("walk4", "e", "e", "en", "e"),
        ("walk4", "e", "e", "es", "e"),
        ("walk4", "w", "w", "en", "w"),
        ("walk4", "w", "w", "es", "w"),
        ("walk4", "w", "e", "e", "en"),
        ("walk4", "e", "e", "e", "es"),
        ("walk4", "w", "w", "w", "en"),
        ("walk4", "w", "w", "w", "es"),
        ("walk4", "e", "en", "w", "w"),
        ("walk4", "e", "es", "w", "w"),
        ("walk4", "w", "en", "e", "e"),
        ("walk4", "w", "es", "e", "e"),
        ("walk4", "e", "e", "en", "w"),
        ("walk4", "e", "e", "es", "w"),
        ("walk4", "w", "w", "en", "e"),
        ("walk4", "w", "w", "es", "e"),
        ("walk4", "wn", "e", "e", "e"),
        ("walk4", "ws", "e", "e", "e"),
        ("walk4", "wn", "w", "w", "w"),
        ("walk4", "ws", "w", "w", "w"),
        ("walk4", "e", "wn", "e", "e"),
        ("walk4", "e", "ws", "e", "e"),
        ("walk4", "w", "wn", "w", "w"),
        ("walk4", "w", "ws", "w", "w"),
        ("walk4", "e", "e", "wn", "e"),
        ("walk4", "e", "e", "ws", "e"),
        ("walk4", "w", "w", "wn", "w"),
        ("walk4", "w", "w", "ws", "w"),
        ("walk4", "w", "e", "e", "wn"),
        ("walk4", "e", "e", "e", "ws"),
        ("walk4", "w", "w", "w", "wn"),
        ("walk4", "w", "w", "w", "ws"),
        ("walk4", "e", "wn", "w", "w"),
        ("walk4", "e", "ws", "w", "w"),
        ("walk4", "w", "wn", "e", "e"),
        ("walk4", "w", "ws", "e", "e"),
        ("walk4", "e", "e", "wn", "w"),
        ("walk4", "e", "e", "ws", "w"),
        ("walk4", "w", "w", "wn", "e"),
        ("walk4", "w", "w", "ws", "e"),
        ("walk4", "1s", "e", "e", "e"),
        ("walk4", "2s", "e", "e", "e"),
        ("walk4", "3s", "e", "e", "e"),
        ("walk4", "4s", "e", "e", "e"),
        ("walk4", "5s", "e", "e", "e"),
        ("walk4", "6s", "e", "e", "e"),
        ("walk4", "7s", "e", "e", "e"),
        ("walk4", "8s", "e", "e", "e"),
        ("walk4", "1s", "w", "w", "w"),
        ("walk4", "2s", "w", "w", "w"),
        ("walk4", "3s", "w", "w", "w"),
        ("walk4", "4s", "w", "w", "w"),
        ("walk4", "5s", "w", "w", "w"),
        ("walk4", "6s", "w", "w", "w"),
        ("walk4", "7s", "w", "w", "w"),
        ("walk4", "8s", "w", "w", "w"),
    ],

    "eagle": [
        ("walk", "n"),
        ("walk", "s"),
        ("walk", "en"),
        ("walk", "wn"),
        ("walk", "es"),
        ("walk", "ws"),
        ("walk", "1s"),
        ("walk", "2s"),
        ("walk", "3s"),
        ("walk", "4s"),
        ("walk", "5s"),
        ("walk", "6s"),
        ("walk", "7s"),
        ("walk", "8s"),
        ("walk", "ne"),
        ("walk", "se"),
        ("walk", "nw"),
        ("walk", "sw"),
        ("fly2", "ne", "e"),
        ("fly2", "se", "e"),
        ("fly2", "nw", "w"),
        ("fly2", "sw", "w"),
        ("fly2", "e", "e"),
        ("fly2", "w", "w"),
        ("fly2", "en", "e"),
        ("fly2", "wn", "w"),
        ("fly2", "es", "e"),
        ("fly2", "ws", "w"),
        ("fly2", "e", "en"),
        ("fly2", "w", "wn"),
        ("fly2", "e", "es"),
        ("fly2", "w", "ws"),
        ("fly2", "e", "ne"),
        ("fly2", "w", "nw"),
        ("fly2", "e", "se"),
        ("fly2", "w", "sw"),
    ],

    "roc": [
        ("walk", "n"),
        ("walk", "s"),
        ("walk", "en"),
        ("walk", "wn"),
        ("walk", "es"),
        ("walk", "ws"),
        ("walk", "1s"),
        ("walk", "2s"),
        ("walk", "3s"),
        ("walk", "4s"),
        ("walk", "5s"),
        ("walk", "6s"),
        ("walk", "7s"),
        ("walk", "8s"),
        ("walk", "ne"),
        ("walk", "se"),
        ("walk", "nw"),
        ("walk", "sw"),
        ("fly2", "ne", "e"),
        ("fly2", "se", "e"),
        ("fly2", "nw", "w"),
        ("fly2", "sw", "w"),
        ("fly2", "en", "e"),
        ("fly2", "wn", "w"),
        ("fly2", "es", "e"),
        ("fly2", "ws", "w"),
        ("fly2", "e", "en"),
        ("fly2", "w", "wn"),
        ("fly2", "e", "es"),
        ("fly2", "w", "ws"),
        ("fly2", "e", "ne"),
        ("fly2", "w", "nw"),
        ("fly2", "e", "se"),
        ("fly2", "w", "sw"),
        ("fly3", "e", "e", "e"),
        ("fly3", "w", "w", "w"),
        ("fly3", "ne", "e", "e"),
        ("fly3", "se", "e", "e"),
        ("fly3", "nw", "w", "w"),
        ("fly3", "sw", "w", "w"),
        ("fly3", "en", "e", "e"),
        ("fly3", "wn", "w", "w"),
        ("fly3", "es", "e", "e"),
        ("fly3", "ws", "w", "w"),
        ("fly3", "e", "e", "en"),
        ("fly3", "w", "w", "wn"),
        ("fly3", "e", "e", "es"),
        ("fly3", "w", "w", "ws"),
        ("fly3", "e", "e", "ne"),
        ("fly3", "w", "w", "nw"),
        ("fly3", "e", "e", "se"),
        ("fly3", "w", "w", "sw"),
        ("fly3", "en", "e", "e"),
        ("fly3", "se", "e", "e"),
        ("fly3", "wn", "w", "w"),
        ("fly3", "ws", "w", "w"),
        ("fly3", "n", "e", "e"),
        ("fly3", "n", "w", "w"),
    ],

    "wizard": [
        ("fly2", "en", "n"),
        ("fly2", "wn", "n"),
        ("fly3", "en", "n", "n"),
        ("fly3", "wn", "n", "n"), 
        
        ("walk", "n"),
        ("walk", "s"),
        ("walk", "e"),
        ("walk", "w"),
        ("walk", "en"),
        ("walk", "wn"),
        ("walk", "es"),
        ("walk", "ws"),
        ("walk", "1s"),
        ("walk", "2s"),
        ("walk", "3s"),
        ("walk", "4s"),
        ("walk", "5s"),
        ("walk", "6s"),
        ("walk", "7s"),
        ("walk", "8s"),
        ("walk", "ne"),
        ("walk", "se"),
        ("walk", "nw"),
        ("walk", "sw"),
        ("fly2", "ne", "e"),
        ("fly2", "se", "e"),
        ("fly2", "nw", "w"),
        ("fly2", "sw", "w"),
        ("fly2", "en", "e"),
        ("fly2", "wn", "w"),
        ("fly2", "es", "e"),
        ("fly2", "ws", "w"),
        ("fly2", "e", "en"),
        ("fly2", "w", "wn"),
        ("fly2", "e", "es"),
        ("fly2", "w", "ws"),
        ("fly2", "e", "ne"),
        ("fly2", "w", "nw"),
        ("fly2", "e", "se"),
        ("fly2", "w", "sw"),
        ("fly2", "e", "e"),
        ("fly2", "w", "w"),
        ("fly3", "e", "e", "e"),
        ("fly3", "w", "w", "w"),
        ("fly3", "ne", "e", "e"),
        ("fly3", "se", "e", "e"),
        ("fly3", "nw", "w", "w"),
        ("fly3", "sw", "w", "w"),
        ("fly3", "en", "e", "e"),
        ("fly3", "wn", "w", "w"),
        ("fly3", "es", "e", "e"),
        ("fly3", "ws", "w", "w"),
        ("fly3", "e", "e", "en"),
        ("fly3", "w", "w", "wn"),
        ("fly3", "e", "e", "es"),
        ("fly3", "w", "w", "ws"),
        ("fly3", "e", "e", "ne"),
        ("fly3", "w", "w", "nw"),
        ("fly3", "e", "e", "se"),
        ("fly3", "w", "w", "sw"),
        ("fly3", "en", "e", "e"),
        ("fly3", "se", "e", "e"),
        ("fly3", "wn", "w", "w"),
        ("fly3", "ws", "w", "w"),

        ("fly2", "n", "n"),
        ("fly2", "s", "s"),
        ("fly2", "n", "ne"),
        ("fly2", "n", "nw"),
        ("fly2", "s", "se"),
        ("fly2", "s", "sw"),
        ("fly2", "n", "en"),
        ("fly2", "n", "wn"),
        ("fly2", "s", "es"),
        ("fly2", "s", "ws"),
        ("fly2", "ne", "ne"),
        ("fly2", "nw", "nw"),
        ("fly2", "sw", "sw"),
        ("fly2", "se", "se"),
        ("fly2", "ne", "en"),
        ("fly2", "nw", "wn"),
        ("fly2", "sw", "ws"),
        ("fly2", "se", "es"),
        ("fly2", "en", "ne"),
        ("fly2", "wn", "nw"),
        ("fly2", "ws", "sw"),
        ("fly2", "es", "se"),

        ("fly3", "ne", "ne", "ne"),
        ("fly3", "se", "se", "se"),
        ("fly3", "nw", "nw", "nw"),
        ("fly3", "sw", "sw", "sw"),
        ("fly3", "ne", "en", "ne"),
        ("fly3", "se", "es", "se"),
        ("fly3", "nw", "wn", "nw"),
        ("fly3", "sw", "ws", "sw"),
        ("fly3", "es", "se", "es"),
        ("fly3", "wn", "nw", "wn"),
        ("fly3", "ws", "sw", "ws"),
        ("fly3", "en", "ne", "n"),
        ("fly3", "wn", "nw", "n"),
        ("fly3", "n", "n", "n"),
        ("fly3", "s", "s", "s"),
        ("fly3", "n", "en", "n"),
        ("fly3", "n", "wn", "n"),
        ("fly3", "s", "es", "s"),
        ("fly3", "s", "ws", "s"),
        ("fly3", "n", "n", "ne"),
        ("fly3", "n", "n", "nw"),
        ("fly3", "s", "s", "se"),
        ("fly3", "s", "s", "sw"),
        ("fly3", "n", "ne", "ne"),
        ("fly3", "n", "nw", "nw"),
        ("fly3", "s", "se", "se"),
        ("fly3", "s", "sw", "sw"),
        ("fly3", "ne", "ne", "n"),
        ("fly3", "nw", "nw", "n"),
        ("fly3", "se", "se", "s"),
        ("fly3", "sw", "sw", "s"),
        ("fly3", "ne", "se", "se"),
        ("fly3", "nw", "sw", "sw"),
        ("fly3", "ne", "ne", "se"),
        ("fly3", "nw", "nw", "sw"),
        ("fly3", "e", "ne", "ne"),
        ("fly3", "w", "nw", "nw"),
        ("fly3", "e", "se", "se"),
        ("fly3", "w", "sw", "sw"),

        ("fly3", "1s", "es", "s"),
        ("fly3", "1s", "ws", "s"),
        ("fly3", "2s", "es", "s"),
        ("fly3", "2s", "ws", "s"),
        ("fly3", "3s", "es", "s"),
        ("fly3", "3s", "ws", "s"),
        ("fly3", "4s", "es", "s"),
        ("fly3", "4s", "ws", "s"),
        ("fly3", "5s", "es", "s"),
        ("fly3", "5s", "ws", "s"),
        ("fly3", "6s", "es", "s"),
        ("fly3", "6s", "ws", "s"),
        ("fly3", "7s", "es", "s"),
        ("fly3", "7s", "ws", "s"),
        ("fly3", "8s", "es", "s"),
        ("fly3", "8s", "ws", "s"),

        ("fly3", "n", "n", "1s"),
        ("fly3", "n", "n", "2s"),
        ("fly3", "n", "n", "3s"),
        ("fly3", "n", "n", "4s"),
        ("fly3", "n", "n", "5s"),
        ("fly3", "n", "n", "6s"),
        ("fly3", "n", "n", "7s"),
        ("fly3", "n", "n", "8s"),

        ("fly3", "n", "1s", "es"),
        ("fly3", "n", "1s", "ws"),
        ("fly3", "n", "2s", "es"),
        ("fly3", "n", "2s", "ws"),
        ("fly3", "n", "3s", "es"),
        ("fly3", "n", "3s", "ws"),
        ("fly3", "n", "4s", "es"),
        ("fly3", "n", "4s", "ws"),
        ("fly3", "n", "5s", "es"),
        ("fly3", "n", "5s", "ws"),
        ("fly3", "n", "6s", "es"),
        ("fly3", "n", "6s", "ws"),
        ("fly3", "n", "7s", "es"),
        ("fly3", "n", "7s", "ws"),
        ("fly3", "n", "8s", "es"),
        ("fly3", "n", "8s", "ws"),

        ("fly3", "se", "s", "s"),
        ("fly3", "sw", "s", "s"),
        ("fly3", "es", "s", "s"),
        ("fly3", "ws", "s", "s"),

        ("fly3", "se", "s", "s"),
        ("fly3", "sw", "s", "s"),
        ("fly3", "es", "s", "ws"),
        ("fly3", "ws", "s", "es"),
        ("fly3", "se", "se", "ws"),
        ("fly3", "sw", "sw", "es"),

        ("fly2", "1s", "es"),
        ("fly2", "1s", "ws"),
        ("fly2", "2s", "es"),
        ("fly2", "2s", "ws"),
        ("fly2", "3s", "es"),
        ("fly2", "3s", "ws"),
        ("fly2", "4s", "es"),
        ("fly2", "4s", "ws"),
        ("fly2", "5s", "es"),
        ("fly2", "5s", "ws"),
        ("fly2", "6s", "es"),
        ("fly2", "6s", "ws"),
        ("fly2", "7s", "es"),
        ("fly2", "7s", "ws"),
        ("fly2", "8s", "es"),
        ("fly2", "8s", "ws"),

        ("fly3", "n", "e", "se"),
        ("fly3", "se", "e", "se"),
        ("fly3", "n", "w", "sw"),
        ("fly3", "n", "w", "sw"),
        ("fly3", "ne", "e", "ne"),
        ("fly3", "se", "e", "se"),
        ("fly3", "nw", "w", "nw"),
        ("fly3", "sw", "w", "sw"),
        ("fly3", "ne", "e", "se"),
        ("fly3", "se", "e", "ne"),
        ("fly3", "nw", "w", "sw"),
        ("fly3", "sw", "w", "nw"),
        ("fly2", "n", "1s"),
        ("fly2", "n", "2s"),
        ("fly2", "n", "3s"),
        ("fly2", "n", "4s"),
        ("fly2", "n", "5s"),
        ("fly2", "n", "6s"),
        ("fly2", "n", "7s"),
        ("fly2", "n", "8s"),
        ("fly3", "ne", "ne", "e"),
        ("fly3", "se", "se", "e"),
        ("fly3", "nw", "nw", "w"),
        ("fly3", "sw", "sw", "w"),
        ("fly3", "ne", "e", "es"),
        ("fly3", "nw", "w", "ws"),
        ("fly2", "se", "s"),
        ("fly2", "sw", "s"),
        ("fly2", "es", "s"),
        ("fly2", "ws", "s"),
        ("fly3", "e", "en", "ne"),
        ("fly3", "w", "wn", "nw"),
        ("fly3", "e", "ne", "en"),
        ("fly3", "w", "nw", "wn"),
        ("fly3", "es", "e", "se"),
        ("fly3", "ws", "w", "sw"),
        ("fly3", "se", "es", "e"),
        ("fly3", "sw", "ws", "w"),
    ],

    "dragon": [
        ("walk", "n"),
        ("walk", "s"),
        ("walk", "en"),
        ("walk", "wn"),
        ("walk", "es"),
        ("walk", "ws"),
        ("walk", "1s"),
        ("walk", "2s"),
        ("walk", "3s"),
        ("walk", "4s"),
        ("walk", "5s"),
        ("walk", "6s"),
        ("walk", "7s"),
        ("walk", "8s"),
        ("walk", "ne"),
        ("walk", "se"),
        ("walk", "nw"),
        ("walk", "sw"),
        ("fly2", "ne", "e"),
        ("fly2", "se", "e"),
        ("fly2", "nw", "w"),
        ("fly2", "sw", "w"),
        ("fly2", "en", "e"),
        ("fly2", "wn", "w"),
        ("fly2", "es", "e"),
        ("fly2", "ws", "w"),
        ("fly2", "e", "en"),
        ("fly2", "w", "wn"),
        ("fly2", "e", "es"),
        ("fly2", "w", "ws"),
        ("fly2", "e", "ne"),
        ("fly2", "w", "nw"),
        ("fly2", "e", "se"),
        ("fly2", "w", "sw"),
        ("fly4", "e", "e", "e", "e"),
        ("fly4", "w", "w", "w", "w"),
        ("fly3", "ne", "e", "e"),
        ("fly3", "se", "e", "e"),
        ("fly3", "nw", "w", "w"),
        ("fly3", "sw", "w", "w"),
        ("fly3", "en", "e", "e"),
        ("fly3", "wn", "w", "w"),
        ("fly3", "es", "e", "e"),
        ("fly3", "ws", "w", "w"),
        ("fly3", "e", "e", "en"),
        ("fly3", "w", "w", "wn"),
        ("fly3", "e", "e", "es"),
        ("fly3", "w", "w", "ws"),
        ("fly3", "e", "e", "ne"),
        ("fly3", "w", "w", "nw"),
        ("fly3", "e", "e", "se"),
        ("fly3", "w", "w", "sw"),
        ("fly3", "en", "e", "e"),
        ("fly3", "se", "e", "e"),
        ("fly3", "wn", "w", "w"),
        ("fly3", "ws", "w", "w"),
        ("fly4", "ne", "e", "e", "e"),
        ("fly4", "se", "e", "e", "e"),
        ("fly4", "nw", "w", "w", "w"),
        ("fly4", "sw", "w", "w", "w"),
        ("fly4", "en", "e", "e", "e"),
        ("fly4", "wn", "w", "w", "w"),
        ("fly4", "es", "e", "e", "e"),
        ("fly4", "ws", "w", "w", "w"),
        ("fly4", "e", "e", "e", "en"),
        ("fly4", "w", "w", "w", "wn"),
        ("fly4", "e", "e", "e", "es"),
        ("fly4", "w", "w", "w", "ws"),
        ("fly4", "e", "e", "e", "ne"),
        ("fly4", "w", "w", "w", "nw"),
        ("fly4", "e", "e", "e", "se"),
        ("fly4", "w", "w", "w", "sw"),
        ("fly4", "en", "e", "e", "e"),
        ("fly4", "se", "e", "e", "e"),
        ("fly4", "wn", "w", "w", "w"),
        ("fly4", "ws", "w", "w", "w"),
        ("fly4", "n", "e", "e", "e"),
        ("fly4", "n", "w", "w", "w"),
        ("fly4", "e", "e", "e", "s"),
        ("fly4", "w", "w", "w", "s"),
    ],

    
    "fairy": [
        ("walk", "n"),
        ("walk", "s"),
        ("walk", "e"),
        ("walk", "w"),
        ("walk", "en"),
        ("walk", "wn"),
        ("walk", "es"),
        ("walk", "ws"),
        ("walk", "1s"),
        ("walk", "2s"),
        ("walk", "3s"),
        ("walk", "4s"),
        ("walk", "5s"),
        ("walk", "6s"),
        ("walk", "7s"),
        ("walk", "8s"),
        ("walk", "ne"),
        ("walk", "se"),
        ("walk", "nw"),
        ("walk", "sw"),
        ("fly2", "ne", "e"),
        ("fly2", "se", "e"),
        ("fly2", "nw", "w"),
        ("fly2", "sw", "w"),
        ("fly2", "en", "e"),
        ("fly2", "wn", "w"),
        ("fly2", "es", "e"),
        ("fly2", "ws", "w"),
        ("fly2", "e", "en"),
        ("fly2", "w", "wn"),
        ("fly2", "e", "es"),
        ("fly2", "w", "ws"),
        ("fly2", "e", "ne"),
        ("fly2", "w", "nw"),
        ("fly2", "e", "se"),
        ("fly2", "w", "sw"),
        ("fly2", "e", "e"),
        ("fly2", "w", "w"),


        ("fly2", "n", "n"),
        ("fly2", "s", "s"),
        ("fly2", "n", "ne"),
        ("fly2", "n", "nw"),
        ("fly2", "s", "se"),
        ("fly2", "s", "sw"),
        ("fly2", "n", "en"),
        ("fly2", "n", "wn"),
        ("fly2", "s", "es"),
        ("fly2", "s", "ws"),
        ("fly2", "ne", "ne"),
        ("fly2", "nw", "nw"),
        ("fly2", "sw", "sw"),
        ("fly2", "se", "se"),
        ("fly2", "ne", "en"),
        ("fly2", "nw", "wn"),
        ("fly2", "sw", "ws"),
        ("fly2", "se", "es"),
        ("fly2", "en", "ne"),
        ("fly2", "wn", "nw"),
        ("fly2", "ws", "sw"),
        ("fly2", "es", "se"),

        ("fly2", "1s", "es"),
        ("fly2", "1s", "ws"),
        ("fly2", "2s", "es"),
        ("fly2", "2s", "ws"),
        ("fly2", "3s", "es"),
        ("fly2", "3s", "ws"),
        ("fly2", "4s", "es"),
        ("fly2", "4s", "ws"),
        ("fly2", "5s", "es"),
        ("fly2", "5s", "ws"),
        ("fly2", "6s", "es"),
        ("fly2", "6s", "ws"),
        ("fly2", "7s", "es"),
        ("fly2", "7s", "ws"),
        ("fly2", "8s", "es"),
        ("fly2", "8s", "ws"),

        ("fly2", "n", "1s"),
        ("fly2", "n", "2s"),
        ("fly2", "n", "3s"),
        ("fly2", "n", "4s"),
        ("fly2", "n", "5s"),
        ("fly2", "n", "6s"),
        ("fly2", "n", "7s"),
        ("fly2", "n", "8s"),

        ("fly2", "se", "s"),
        ("fly2", "sw", "s"),
        ("fly2", "es", "s"),
        ("fly2", "ws", "s"),

        ("fly2", "en", "n"),
        ("fly2", "wn", "n"),

        ("fly2", "ne", "se"),
        ("fly2", "nw", "sw"),
    ],

    "magus": [
        ("fly2", "en", "n"),
        ("fly2", "wn", "n"),
        ("fly3", "en", "n", "n"),
        ("fly3", "wn", "n", "n"),      
        
        ("walk", "n"),
        ("walk", "s"),
        ("walk", "e"),
        ("walk", "w"),
        ("walk", "en"),
        ("walk", "wn"),
        ("walk", "es"),
        ("walk", "ws"),
        ("walk", "1s"),
        ("walk", "2s"),
        ("walk", "3s"),
        ("walk", "4s"),
        ("walk", "5s"),
        ("walk", "6s"),
        ("walk", "7s"),
        ("walk", "8s"),
        ("walk", "ne"),
        ("walk", "se"),
        ("walk", "nw"),
        ("walk", "sw"),
        ("fly2", "ne", "e"),
        ("fly2", "se", "e"),
        ("fly2", "nw", "w"),
        ("fly2", "sw", "w"),
        ("fly2", "en", "e"),
        ("fly2", "wn", "w"),
        ("fly2", "es", "e"),
        ("fly2", "ws", "w"),
        ("fly2", "e", "en"),
        ("fly2", "w", "wn"),
        ("fly2", "e", "es"),
        ("fly2", "w", "ws"),
        ("fly2", "e", "ne"),
        ("fly2", "w", "nw"),
        ("fly2", "e", "se"),
        ("fly2", "w", "sw"),
        ("fly2", "e", "e"),
        ("fly2", "w", "w"),
        ("fly3", "e", "e", "e"),
        ("fly3", "w", "w", "w"),
        ("fly3", "ne", "e", "e"),
        ("fly3", "se", "e", "e"),
        ("fly3", "nw", "w", "w"),
        ("fly3", "sw", "w", "w"),
        ("fly3", "en", "e", "e"),
        ("fly3", "wn", "w", "w"),
        ("fly3", "es", "e", "e"),
        ("fly3", "ws", "w", "w"),
        ("fly3", "e", "e", "en"),
        ("fly3", "w", "w", "wn"),
        ("fly3", "e", "e", "es"),
        ("fly3", "w", "w", "ws"),
        ("fly3", "e", "e", "ne"),
        ("fly3", "w", "w", "nw"),
        ("fly3", "e", "e", "se"),
        ("fly3", "w", "w", "sw"),
        ("fly3", "en", "e", "e"),
        ("fly3", "se", "e", "e"),
        ("fly3", "wn", "w", "w"),
        ("fly3", "ws", "w", "w"),

        ("fly2", "n", "n"),
        ("fly2", "s", "s"),
        ("fly2", "n", "ne"),
        ("fly2", "n", "nw"),
        ("fly2", "s", "se"),
        ("fly2", "s", "sw"),
        ("fly2", "n", "en"),
        ("fly2", "n", "wn"),
        ("fly2", "s", "es"),
        ("fly2", "s", "ws"),
        ("fly2", "ne", "ne"),
        ("fly2", "nw", "nw"),
        ("fly2", "sw", "sw"),
        ("fly2", "se", "se"),
        ("fly2", "ne", "en"),
        ("fly2", "nw", "wn"),
        ("fly2", "sw", "ws"),
        ("fly2", "se", "es"),
        ("fly2", "en", "ne"),
        ("fly2", "wn", "nw"),
        ("fly2", "ws", "sw"),
        ("fly2", "es", "se"),

        ("fly3", "ne", "ne", "ne"),
        ("fly3", "se", "se", "se"),
        ("fly3", "nw", "nw", "nw"),
        ("fly3", "sw", "sw", "sw"),
        ("fly3", "ne", "en", "ne"),
        ("fly3", "se", "es", "se"),
        ("fly3", "nw", "wn", "nw"),
        ("fly3", "sw", "ws", "sw"),
        ("fly3", "es", "se", "es"),
        ("fly3", "wn", "nw", "wn"),
        ("fly3", "ws", "sw", "ws"),
        ("fly3", "en", "ne", "n"),
        ("fly3", "wn", "nw", "n"),
        ("fly3", "n", "n", "n"),
        ("fly3", "s", "s", "s"),
        ("fly3", "n", "en", "n"),
        ("fly3", "n", "wn", "n"),
        ("fly3", "s", "es", "s"),
        ("fly3", "s", "ws", "s"),
        ("fly3", "n", "n", "ne"),
        ("fly3", "n", "n", "nw"),
        ("fly3", "s", "s", "se"),
        ("fly3", "s", "s", "sw"),
        ("fly3", "n", "ne", "ne"),
        ("fly3", "n", "nw", "nw"),
        ("fly3", "s", "se", "se"),
        ("fly3", "s", "sw", "sw"),
        ("fly3", "ne", "ne", "n"),
        ("fly3", "nw", "nw", "n"),
        ("fly3", "se", "se", "s"),
        ("fly3", "sw", "sw", "s"),
        ("fly3", "ne", "se", "se"),
        ("fly3", "nw", "sw", "sw"),
        ("fly3", "ne", "ne", "se"),
        ("fly3", "nw", "nw", "sw"),
        ("fly3", "e", "ne", "ne"),
        ("fly3", "w", "nw", "nw"),
        ("fly3", "e", "se", "se"),
        ("fly3", "w", "sw", "sw"),

        ("fly3", "1s", "es", "s"),
        ("fly3", "1s", "ws", "s"),
        ("fly3", "2s", "es", "s"),
        ("fly3", "2s", "ws", "s"),
        ("fly3", "3s", "es", "s"),
        ("fly3", "3s", "ws", "s"),
        ("fly3", "4s", "es", "s"),
        ("fly3", "4s", "ws", "s"),
        ("fly3", "5s", "es", "s"),
        ("fly3", "5s", "ws", "s"),
        ("fly3", "6s", "es", "s"),
        ("fly3", "6s", "ws", "s"),
        ("fly3", "7s", "es", "s"),
        ("fly3", "7s", "ws", "s"),
        ("fly3", "8s", "es", "s"),
        ("fly3", "8s", "ws", "s"),

        ("fly3", "n", "n", "1s"),
        ("fly3", "n", "n", "2s"),
        ("fly3", "n", "n", "3s"),
        ("fly3", "n", "n", "4s"),
        ("fly3", "n", "n", "5s"),
        ("fly3", "n", "n", "6s"),
        ("fly3", "n", "n", "7s"),
        ("fly3", "n", "n", "8s"),

        ("fly3", "n", "1s", "es"),
        ("fly3", "n", "1s", "ws"),
        ("fly3", "n", "2s", "es"),
        ("fly3", "n", "2s", "ws"),
        ("fly3", "n", "3s", "es"),
        ("fly3", "n", "3s", "ws"),
        ("fly3", "n", "4s", "es"),
        ("fly3", "n", "4s", "ws"),
        ("fly3", "n", "5s", "es"),
        ("fly3", "n", "5s", "ws"),
        ("fly3", "n", "6s", "es"),
        ("fly3", "n", "6s", "ws"),
        ("fly3", "n", "7s", "es"),
        ("fly3", "n", "7s", "ws"),
        ("fly3", "n", "8s", "es"),
        ("fly3", "n", "8s", "ws"),

        ("fly3", "se", "s", "s"),
        ("fly3", "sw", "s", "s"),
        ("fly3", "es", "s", "s"),
        ("fly3", "ws", "s", "s"),

        ("fly3", "se", "s", "s"),
        ("fly3", "sw", "s", "s"),
        ("fly3", "es", "s", "ws"),
        ("fly3", "ws", "s", "es"),
        ("fly3", "se", "se", "ws"),
        ("fly3", "sw", "sw", "es"),

        ("fly2", "1s", "es"),
        ("fly2", "1s", "ws"),
        ("fly2", "2s", "es"),
        ("fly2", "2s", "ws"),
        ("fly2", "3s", "es"),
        ("fly2", "3s", "ws"),
        ("fly2", "4s", "es"),
        ("fly2", "4s", "ws"),
        ("fly2", "5s", "es"),
        ("fly2", "5s", "ws"),
        ("fly2", "6s", "es"),
        ("fly2", "6s", "ws"),
        ("fly2", "7s", "es"),
        ("fly2", "7s", "ws"),
        ("fly2", "8s", "es"),
        ("fly2", "8s", "ws"),

        ("fly3", "n", "e", "se"),
        ("fly3", "se", "e", "se"),
        ("fly3", "n", "w", "sw"),
        ("fly3", "n", "w", "sw"),
        ("fly3", "ne", "e", "ne"),
        ("fly3", "se", "e", "se"),
        ("fly3", "nw", "w", "nw"),
        ("fly3", "sw", "w", "sw"),
        ("fly3", "ne", "e", "se"),
        ("fly3", "se", "e", "ne"),
        ("fly3", "nw", "w", "sw"),
        ("fly3", "sw", "w", "nw"),
        ("fly2", "n", "1s"),
        ("fly2", "n", "2s"),
        ("fly2", "n", "3s"),
        ("fly2", "n", "4s"),
        ("fly2", "n", "5s"),
        ("fly2", "n", "6s"),
        ("fly2", "n", "7s"),
        ("fly2", "n", "8s"),
        ("fly3", "ne", "ne", "e"),
        ("fly3", "se", "se", "e"),
        ("fly3", "nw", "nw", "w"),
        ("fly3", "sw", "sw", "w"),
        ("fly3", "ne", "e", "es"),
        ("fly3", "nw", "w", "ws"),
        ("fly2", "se", "s"),
        ("fly2", "sw", "s"),
        ("fly2", "es", "s"),
        ("fly2", "ws", "s"),
        ("fly3", "e", "en", "ne"),
        ("fly3", "w", "wn", "nw"),
        ("fly3", "e", "ne", "en"),
        ("fly3", "w", "nw", "wn"),
        ("fly3", "es", "e", "se"),
        ("fly3", "ws", "w", "sw"),
        ("fly3", "se", "es", "e"),
        ("fly3", "sw", "ws", "w"),


        ("fly4", "e", "e", "e", "e"),
        ("fly4", "w", "w", "w", "w"),
        ("fly4", "n", "e", "e", "e"),
        ("fly4", "n", "w", "w", "w"),
        ("fly4", "ne", "e", "e", "e"),
        ("fly4", "se", "e", "e", "e"),
        ("fly4", "nw", "w", "w", "w"),
        ("fly4", "sw", "w", "w", "w"),
        ("fly4", "en", "e", "e", "e"),
        ("fly4", "wn", "w", "w", "w"),
        ("fly4", "es", "e", "e", "e"),
        ("fly4", "ws", "w", "w", "w"),
        ("fly4", "e", "e", "e", "en"),
        ("fly4", "w", "w", "w", "wn"),
        ("fly4", "e", "e", "e", "es"),
        ("fly4", "w", "w", "w", "ws"),
        ("fly4", "e", "e", "e", "ne"),
        ("fly4", "w", "w", "w", "nw"),
        ("fly4", "e", "e", "e", "se"),
        ("fly4", "w", "w", "w", "sw"),
        ("fly4", "en", "e", "e", "e"),
        ("fly4", "se", "e", "e", "e"),
        ("fly4", "wn", "w", "w", "w"),
        ("fly4", "ws", "w", "w", "w"),
        ("fly4", "ne", "ne", "ne", "ne"),
        ("fly4", "se", "se", "se", "se"),
        ("fly4", "nw", "nw", "nw", "nw"),
        ("fly4", "sw", "sw", "sw", "sw"),
        ("fly4", "ne", "en", "ne", "n"),
        ("fly4", "se", "es", "se", "e"),
        ("fly4", "nw", "wn", "nw", "n"),
        ("fly4", "sw", "ws", "sw", "s"),
        ("fly4", "es", "se", "es", "se"),
        ("fly4", "wn", "nw", "wn", "n"),
        ("fly4", "ws", "sw", "ws", "sw"),
        ("fly4", "en", "ne", "n", "n"),
        ("fly4", "wn", "nw", "n", "n"),
        ("fly4", "n", "n", "n", "n"),
        ("fly4", "s", "s", "s", "s"),

        ("fly4", "n", "en", "n", "n"),
        ("fly4", "n", "wn", "n", "n"),
        ("fly4", "n", "n", "n", "ne"),
        ("fly4", "n", "n", "n", "nw"),
        ("fly4", "s", "s", "s", "se"),
        ("fly4", "s", "s", "s", "sw"),
        ("fly4", "n", "ne", "ne", "ne"),
        ("fly4", "n", "nw", "nw", "nw"),
        ("fly4", "s", "se", "se", "se"),
        ("fly4", "s", "sw", "sw", "sw"),
        ("fly4", "ne", "ne", "ne", "n"),
        ("fly4", "nw", "nw", "nw", "n"),
        ("fly4", "se", "se", "se", "s"),
        ("fly4", "sw", "sw", "sw", "s"),
        ("fly4", "ne", "se", "se", "se"),
        ("fly4", "nw", "sw", "sw", "sw"),
        ("fly4", "ne", "ne", "ne", "se"),
        ("fly4", "nw", "nw", "nw", "sw"),
        ("fly4", "e", "ne", "ne", "ne"),
        ("fly4", "w", "nw", "nw", "nw"),
        ("fly4", "e", "se", "se", "se"),
        ("fly4", "w", "sw", "sw", "sw"),

        ("fly4", "1s", "es", "s", "s"),
        ("fly4", "1s", "ws", "s", "s"),
        ("fly4", "2s", "es", "s", "s"),
        ("fly4", "2s", "ws", "s", "s"),
        ("fly4", "3s", "es", "s", "s"),
        ("fly4", "3s", "ws", "s", "s"),
        ("fly4", "4s", "es", "s", "s"),
        ("fly4", "4s", "ws", "s", "s"),
        ("fly4", "5s", "es", "s", "s"),
        ("fly4", "5s", "ws", "s", "s"),
        ("fly4", "6s", "es", "s", "s"),
        ("fly4", "6s", "ws", "s", "s"),
        ("fly4", "7s", "es", "s", "s"),
        ("fly4", "7s", "ws", "s", "s"),
        ("fly4", "8s", "es", "s", "s"),
        ("fly4", "8s", "ws", "s", "s"),

        ("fly4", "n", "n", "1s", "s"),
        ("fly4", "n", "n", "2s", "s"),
        ("fly4", "n", "n", "3s", "s"),
        ("fly4", "n", "n", "4s", "s"),
        ("fly4", "n", "n", "5s", "s"),
        ("fly4", "n", "n", "6s", "s"),
        ("fly4", "n", "n", "7s", "s"),
        ("fly4", "n", "n", "8s", "s"),

        ("fly4", "n", "n", "n", "1s"),
        ("fly4", "n", "n", "n", "2s"),
        ("fly4", "n", "n", "n", "3s"),
        ("fly4", "n", "n", "n", "4s"),
        ("fly4", "n", "n", "n", "5s"),
        ("fly4", "n", "n", "n", "6s"),
        ("fly4", "n", "n", "n", "7s"),
        ("fly4", "n", "n", "n", "8s"),


        ("fly4", "n", "n", "1s", "es"),
        ("fly4", "n", "n", "1s", "ws"),
        ("fly4", "n", "n", "2s", "es"),
        ("fly4", "n", "n", "2s", "ws"),
        ("fly4", "n", "n", "3s", "es"),
        ("fly4", "n", "n", "3s", "ws"),
        ("fly4", "n", "n", "4s", "es"),
        ("fly4", "n", "n", "4s", "ws"),
        ("fly4", "n", "n", "5s", "es"),
        ("fly4", "n", "n", "5s", "ws"),
        ("fly4", "n", "n", "6s", "es"),
        ("fly4", "n", "n", "6s", "ws"),
        ("fly4", "n", "n", "7s", "es"),
        ("fly4", "n", "n", "7s", "ws"),
        ("fly4", "n", "n", "8s", "es"),
        ("fly4", "n", "n", "8s", "ws"),

        ("fly4", "n", "1s", "es", "s"),
        ("fly4", "n", "1s", "ws", "s"),
        ("fly4", "n", "2s", "es", "s"),
        ("fly4", "n", "2s", "ws", "s"),
        ("fly4", "n", "3s", "es", "s"),
        ("fly4", "n", "3s", "ws", "s"),
        ("fly4", "n", "4s", "es", "s"),
        ("fly4", "n", "4s", "ws", "s"),
        ("fly4", "n", "5s", "es", "s"),
        ("fly4", "n", "5s", "ws", "s"),
        ("fly4", "n", "6s", "es", "s"),
        ("fly4", "n", "6s", "ws", "s"),
        ("fly4", "n", "7s", "es", "s"),
        ("fly4", "n", "7s", "ws", "s"),
        ("fly4", "n", "8s", "es", "s"),
        ("fly4", "n", "8s", "ws", "s"),

        ("fly4", "se", "s", "s", "s"),
        ("fly4", "sw", "s", "s", "s"),
        ("fly4", "es", "s", "s", "s"),
        ("fly4", "ws", "s", "s", "s"),

        ("fly2", "s3", "s"),
        ("fly4", "se", "se", "ws", "s"),
        ("fly4", "sw", "sw", "es", "s"),
        ("fly2", "n3", "n"),
        ("fly2", "n", "n3"),

        ("fly3", "n", "e", "se"),
        ("fly3", "se", "e", "se"),
        ("fly3", "n", "w", "sw"),
        ("fly3", "n", "w", "sw"),


        ("fly4", "ne", "e", "ne", "ne"),
        ("fly4", "se", "e", "se", "es"),
        ("fly4", "nw", "w", "nw", "nw"),
        ("fly4", "sw", "w", "sw", "sw"),
        ("fly4", "ne", "e", "se", "se"),
        ("fly4", "nw", "w", "sw", "sw"),

        ("fly2", "1s", "s3"),
        ("fly2", "2s", "s3"),
        ("fly2", "3s", "s3"),
        ("fly2", "4s", "s3"),
        ("fly2", "5s", "s3"),
        ("fly2", "6s", "s3"),
        ("fly2", "7s", "s3"),
        ("fly2", "8s", "s3"),

        ("fly4", "ne", "ne", "e", "e"),
        ("fly4", "se", "se", "e", "e"),
        ("fly4", "nw", "nw", "w", "w"),
        ("fly4", "sw", "sw", "w", "w"),

        ("fly4", "ne", "ne", "ne", "e"),
        ("fly4", "se", "se", "se", "e"),
        ("fly4", "nw", "nw", "nw", "w"),
        ("fly4", "sw", "sw", "sw", "w"),
        ("fly4", "ne", "e",  "e", "es"),
        ("fly4", "nw", "w", "w", "ws"),

        ("fly4", "e", "en", "ne", "e"),
        ("fly4", "w", "wn", "nw", "w"),
        ("fly4", "e", "ne", "en", "ne"),
        ("fly4", "w", "nw", "wn", "nw"),
        ("fly4", "e", "ne", "en", "ne"),
        ("fly4", "w", "nw", "wn", "nw"),
        ("fly4", "es", "e", "se", "e"),
        ("fly4", "ws", "w", "sw", "w"),
        ("fly4", "se", "es", "e", "e"),
        ("fly4", "sw", "ws", "w", "w"),
        
        ("fly4", "en", "ne", "e", "e"),
        ("fly4", "wn", "nw", "w", "w"),

        ("fly4", "e", "e", "en", "ne"),
        ("fly4", "w", "w", "wn", "nw"),   
        ("fly4", "ne", "e", "se", "s"),
        ("fly4", "nw", "w", "sw", "s"),
        ("fly4", "en", "e", "e", "se"),
        ("fly4", "wn", "w", "w", "sw"),
        ("fly4", "en", "e", "e", "ew"),
        ("fly4", "wn", "w", "w", "nw"),

        ("fly4", "e", "se", "se", "s"),
        ("fly4", "w", "sw", "sw", "s"),
        ("fly4", "e", "e", "se", "se"),
        ("fly4", "w", "w", "sw", "sw"),
        ("fly4", "ne", "se", "se", "se"),
        ("fly4", "nw", "sw", "sw", "sw"),
        ("fly4", "e", "ne", "se", "se"),
        ("fly4", "w", "nw", "sw", "sw"),
        ("fly4", "ne", "se", "se", "s"),
        ("fly4", "ne", "se", "se", "ws"),
        
        ("fly4", "nw", "sw", "sw", "s"),
        ("fly4", "nw", "sw", "sw", "es"),

        ("fly4", "e", "e", "es", "se"),
        ("fly4", "w", "w", "ws", "sw"),  

        
        ("fly4", "se", "e", "e", "se"),
        ("fly4", "ne", "en", "e", "e"),
        ("fly4", "sw", "w", "w", "sw"),
        ("fly4", "nw", "wn", "w", "w"),

        
        ("fly4", "se", "e", "e", "se"),
        ("fly4", "ne", "en", "e", "e"),
        ("fly4", "nw", "nw", "w", "sw"),
        ("fly4", "nw", "n", "w", "w"),
        ("fly4", "ne", "ne", "e", "se"),
        ("fly4", "ne", "n", "e", "e"),
        ("fly4", "ne", "e", "e", "se"),
        ("fly4", "nw", "nw", "w", "ws"),
        ("fly4", "ne", "ne", "e", "es"),

        ("fly4", "ne", "ne", "se", "se"),
        ("fly4", "nw", "nw", "sw", "sw"),

        ("fly4", "n", "ne", "e", "se"),
        ("fly4", "n", "nw", "w", "sw"),
    ],
}
# 다른 기물도 같은 형식으로 나중에 추가 가능
# 예:
# "horse": [("walk2", "n", "e"), ...]


# -----------------------------
# 4. 보드 상태
# -----------------------------
board_setup_5p = {
    "Blue": {
        "pawn":   ["2I"],
        "horse":  ["1E", "1I"],
        "tiger":  ["1D", "1J"],
        "roc":    ["1A"],
        "eagle":  ["1C", "1K"],
        "raven":  ["8E"],
        "wizard": ["1B"],
    },
    "Green": {
        "pawn":   ["3I"],
        "horse":  ["2D", "4J"],
        "tiger":  ["3D", "3J"],
        "roc":    ["3A"],
        "eagle":  ["3C", "3K"],
        "raven":  ["3E"],
        "wizard": ["3B"],
    },
    "Orange": {
        "pawn":   ["6I"],
        "horse":  ["5E", "5I"],
        "tiger":  ["5D", "5J"],
        "roc":    ["5A"],
        "eagle":  ["5C", "5K"],
        "raven":  ["4E"],
        "wizard": ["5B"],
    },
    "Pink": {
        "pawn":   ["7I"],
        "horse":  ["6D", "8J"],
        "tiger":  ["7D", "7J"],
        "roc":    ["7A"],
        "eagle":  ["7C", "7K"],
        "raven":  ["7E"],
        "wizard": ["7B"],
    },
    "Yellow": {
        "pawn":   ["4H", "8H"],
        "horse":  ["2F", "6F"],
        "tiger":  ["4G", "8G"],
        "eagle":  ["1G", "3G", "5G", "7G"],
        "wizard": ["00"],   # normalize to OO
    },
}


# -----------------------------
# 5. 좌표 normalize
# -----------------------------
def normalize_square(s: str) -> str:
    s = s.strip().upper()
    if s == "OO":
        return "00"
    return s


# -----------------------------
# 6. setup -> board map
#   square -> {"color":..., "piece":...}
# -----------------------------
def build_board_map(board_setup):
    board_map = {}
    for color, piece_dict in board_setup.items():
        for piece_name, squares in piece_dict.items():
            for sq in squares:
                sq = normalize_square(sq)
                if sq in board_map:
                    raise ValueError(f"Duplicate occupancy on {sq}")
                board_map[sq] = {
                    "color": color,
                    "piece": piece_name,
                }
    return board_map


# -----------------------------
# 7. 공통 이동 추적
# -----------------------------
def follow_move_sequence(adj, start, sequence, occupied_cells=None, require_empty_intermediate=False):
    if occupied_cells is None:
        occupied_cells = set()

    current = {start}

    for i, link_name in enumerate(sequence):
        is_last = (i == len(sequence) - 1)
        next_cells = set()

        for cell_ in current:
            for nxt in adj.get(link_name, {}).get(cell_, set()):
                if require_empty_intermediate and not is_last and nxt in occupied_cells:
                    continue
                next_cells.add(nxt)

        current = next_cells
        if not current:
            break

    return current

def walk(adj, start, link_name, occupied_cells=None):
    return follow_move_sequence(
        adj, start, [link_name],
        occupied_cells=occupied_cells,
        require_empty_intermediate=False
    )

def walk2(adj, start, a, b, occupied_cells=None):
    return follow_move_sequence(
        adj, start, [a, b],
        occupied_cells=occupied_cells,
        require_empty_intermediate=True
    )

def walk3(adj, start, a, b, c, occupied_cells=None):
    return follow_move_sequence(
        adj, start, [a, b, c],
        occupied_cells=occupied_cells,
        require_empty_intermediate=True
    )

def walk4(adj, start, a, b, c, d, occupied_cells=None):
    return follow_move_sequence(
        adj, start, [a, b, c, d],
        occupied_cells=occupied_cells,
        require_empty_intermediate=True
    )

def fly2(adj, start, a, b, occupied_cells=None):
    return follow_move_sequence(
        adj, start, [a, b],
        occupied_cells=occupied_cells,
        require_empty_intermediate=False
    )

def fly3(adj, start, a, b, c, occupied_cells=None):
    return follow_move_sequence(
        adj, start, [a, b, c],
        occupied_cells=occupied_cells,
        require_empty_intermediate=False
    )

def fly4(adj, start, a, b, c, d, occupied_cells=None):
    return follow_move_sequence(
        adj, start, [a, b, c, d],
        occupied_cells=occupied_cells,
        require_empty_intermediate=False
    )


# -----------------------------
# 8. 기물 하나의 합법 목적지 계산
#   현재는 "소유자 구분 없이 마지막 칸 허용" 대신,
#   자기 칸 점프만 막고, 마지막 칸이 자기 말이면 제외
#   정도로 두는 게 가장 자연스러움
# -----------------------------
def apply_move_def(adj, start, move_def, occupied_cells=None):
    op = move_def[0]
    args = move_def[1:]

    if op == "walk":
        return walk(adj, start, args[0], occupied_cells)
    elif op == "walk2":
        return walk2(adj, start, args[0], args[1], occupied_cells)
    elif op == "walk3":
        return walk3(adj, start, args[0], args[1], args[2], occupied_cells)
    elif op == "walk4":
        return walk4(adj, start, args[0], args[1], args[2], args[3], occupied_cells)
    elif op == "fly2":
        return fly2(adj, start, args[0], args[1], occupied_cells)
    elif op == "fly3":
        return fly3(adj, start, args[0], args[1], args[2], occupied_cells)
    elif op == "fly4":
        return fly4(adj, start, args[0], args[1], args[2], args[3], occupied_cells)
    else:
        raise ValueError(f"Unknown move operator: {op}")

def legal_targets_for_piece(board_map, adj, color, piece_name, start_square):
    start_square = normalize_square(start_square)

    if start_square not in board_map:
        raise ValueError(f"No piece on {start_square}")

    info = board_map[start_square]
    if info["color"] != color or info["piece"] != piece_name:
        raise ValueError(
            f"{start_square} does not contain {color} {piece_name} "
            f"(found {info['color']} {info['piece']})"
        )

    if piece_name not in PIECES:
        raise ValueError(f"No move definition for piece: {piece_name}")

    occupied_cells = set(board_map.keys())
    my_cells = {sq for sq, v in board_map.items() if v["color"] == color}

    result = set()
    for move_def in PIECES[piece_name]:
        result |= apply_move_def(adj, start_square, move_def, occupied_cells)

    result.discard(start_square)

    # 자기 말이 있는 칸은 최종 목적지에서 제외
    

    return sorted(result)


# -----------------------------
# 9. "색 + 기물 + 위치"로 조회하는 도우미
# -----------------------------
def show_legal_moves(board_map, adj, color, piece_name, start_square):
    targets = legal_targets_for_piece(board_map, adj, color, piece_name, start_square)

    print(f"{color} {piece_name} at {normalize_square(start_square)}")
    print("Legal targets:", targets)
    return targets


# -----------------------------
# 10. 초기화
# -----------------------------
links = build_links_from_templates(LINK_TEMPLATES)
links = add_center_links_split(links)
adj = build_adjacency(links)
board_map = build_board_map(board_setup_5p)


# -----------------------------
# 11. 사용 예시
# -----------------------------
show_legal_moves(board_map, adj, "Blue", "pawn", "2I")
show_legal_moves(board_map, adj, "Yellow", "wizard", "00")   # normalize to OO

Blue pawn at 2I
Legal targets: ['1E', '2A', '2E', '2H', '2J']
Yellow wizard at 00
Legal targets: ['1E', '1F', '1G', '1H', '1I', '2E', '2F', '2G', '2H', '2I', '3E', '3F', '3G', '3H', '3I', '4E', '4F', '4G', '4H', '4I', '5E', '5F', '5G', '5H', '5I', '6E', '6F', '6G', '6H', '6I', '7E', '7F', '7G', '7H', '7I', '8E', '8F', '8G', '8H', '8I']


['1E',
 '1F',
 '1G',
 '1H',
 '1I',
 '2E',
 '2F',
 '2G',
 '2H',
 '2I',
 '3E',
 '3F',
 '3G',
 '3H',
 '3I',
 '4E',
 '4F',
 '4G',
 '4H',
 '4I',
 '5E',
 '5F',
 '5G',
 '5H',
 '5I',
 '6E',
 '6F',
 '6G',
 '6H',
 '6I',
 '7E',
 '7F',
 '7G',
 '7H',
 '7I',
 '8E',
 '8F',
 '8G',
 '8H',
 '8I']

In [8]:
from collections import defaultdict

def normalize_square(square: str) -> str:
    square = square.strip().upper()
    if square == "OO":
        return "00"   # 팀 기반 표현에서는 00로 통일
    return square


def board_map_to_team_based(board_map):
    """
    {
      '2I': {'color': 'Blue', 'piece': 'pawn'},
      ...
    }
    ->
    {
      'Blue': {'pawn': ['2I'], ...},
      ...
    }
    """
    result = defaultdict(lambda: defaultdict(list))

    for square, info in board_map.items():
        color = info["color"]
        piece = info["piece"]
        sq = normalize_square(square)
        result[color][piece].append(sq)

    # 정렬 + 일반 dict로 변환
    final = {}
    for color, piece_dict in result.items():
        final[color] = {}
        for piece, squares in piece_dict.items():
            final[color][piece] = sorted(squares)
    return final


def team_based_to_board_map(team_based):
    """
    {
      'Blue': {'pawn': ['2I'], ...},
      ...
    }
    ->
    {
      '2I': {'color': 'Blue', 'piece': 'pawn'},
      ...
    }
    """
    result = {}

    for color, piece_dict in team_based.items():
        for piece, squares in piece_dict.items():
            for square in squares:
                sq = normalize_square(square)
                if sq in result:
                    raise ValueError(f"Duplicate occupancy detected at {sq}")
                result[sq] = {
                    "color": color,
                    "piece": piece
                }

    return result

In [9]:


piece_name_to_code = {
    "pawn": "X1",
    "horse": "X2",
    "tiger": "X3",
    "elephant": "X4",
    "raven": "Y1",
    "eagle": "Y2",
    "roc": "Y3",
    "dragon": "Y4",
    "fairy": "Z2",
    "wizard": "Z3",
    "magus": "Z4",
}

board_setup_2p = {
    "Blue": {
        "pawn":     ["8A", "8D", "2J", "2A"],
        "horse":    ["8J", "2D"],
        "tiger":    ["8K", "2C"],
        "elephant": ["8C", "2K"],
        "roc":      ["1K", "1C"],
        "eagle":    ["2B", "8B"],
        "raven":    ["1D", "1J"],
        "dragon":   ["1A"],
        "wizard":   ["1B"],
    },
    "Orange": {
        "pawn":     ["4A", "4D", "6J", "6A"],
        "horse":    ["4J", "6D"],
        "tiger":    ["4K", "6C"],
        "elephant": ["4C", "6K"],
        "roc":      ["5K", "5C"],
        "eagle":    ["6B", "4B"],
        "raven":    ["5D", "5J"],
        "dragon":   ["5A"],
        "wizard":   ["5B"],
    },
}

board_setup_3p = {
    "Blue": {
        "pawn":   ["8D", "1A", "2J"],
        "horse":  ["8A", "2A"],
        "tiger":  ["8C", "2K"],
        "roc":    ["1K", "1C"],
        "eagle":  ["2B", "8B"],
        "raven":  ["1D", "1J"],
        "wizard": ["1B"],
    },
    "Orange": {
        "pawn":   ["4D", "5A", "6J"],
        "horse":  ["4A", "6A"],
        "tiger":  ["4C", "6K"],
        "roc":    ["5K", "5C"],
        "eagle":  ["6B", "4B"],
        "raven":  ["5D", "5J"],
        "wizard": ["5B"],
    },
    "Green": {
        "pawn":   ["3F", "3H", "7F", "7H"],
        "horse":  ["2G", "6G"],
        "tiger":  ["4G", "8G"],
        "roc":    ["3G", "7G"],
        "eagle":  ["1G", "5G"],
        "raven":  ["3E", "7E"],
        "wizard": ["00"],   # OO -> 00 정규화
    },
}

board_setup_4p = {
    "Blue": {
        "pawn":   ["8C", "8D", "2J", "2K"],
        "horse":  ["1E", "1I"],
        "tiger":  ["1D", "1J"],
        "roc":    ["1C", "1K"],
        "dragon": ["1A"],
        "wizard": ["1B"],
    },
    "Green": {
        "pawn":   ["2C", "2D", "4J", "4K"],
        "horse":  ["3E", "3I"],
        "tiger":  ["3D", "3J"],
        "roc":    ["3C", "3K"],
        "dragon": ["3A"],
        "wizard": ["3B"],
    },
    "Orange": {
        "pawn":   ["4C", "4D", "6J", "6K"],
        "horse":  ["5E", "5I"],
        "tiger":  ["5D", "5J"],
        "roc":    ["5C", "5K"],
        "dragon": ["5A"],
        "wizard": ["5B"],
    },
    "Pink": {
        "pawn":   ["6C", "6D", "8J", "8K"],
        "horse":  ["7E", "7I"],
        "tiger":  ["7D", "7J"],
        "roc":    ["7C", "7K"],
        "dragon": ["7A"],
        "wizard": ["7B"],
    },
}

board_setup_5p = {
    "Blue": {
        "pawn":   ["2I"],
        "horse":  ["1E", "1I"],
        "tiger":  ["1D", "1J"],
        "roc":    ["1A"],
        "eagle":  ["1C", "1K"],
        "raven":  ["8E"],
        "magus": ["1B"],
    },
    "Green": {
        "pawn":   ["3I"],
        "horse":  ["2D", "4J"],
        "tiger":  ["3D", "3J"],
        "roc":    ["3A"],
        "eagle":  ["3C", "3K"],
        "raven":  ["3E"],
        "magus": ["3B"],
    },
    "Orange": {
        "pawn":   ["6I"],
        "horse":  ["5E", "5I"],
        "tiger":  ["5D", "5J"],
        "roc":    ["5A"],
        "eagle":  ["5C", "5K"],
        "raven":  ["4E"],
        "magus": ["5B"],
    },
    "Pink": {
        "pawn":   ["7I"],
        "horse":  ["6D", "8J"],
        "tiger":  ["7D", "7J"],
        "roc":    ["7A"],
        "eagle":  ["7C", "7K"],
        "raven":  ["7E"],
        "magus": ["7B"],
    },
    "Yellow": {
        "pawn":   ["4H", "8H"],
        "horse":  ["2F", "6F"],
        "tiger":  ["4G", "8G"],
        "eagle":  ["1G", "3G", "5G", "7G"],
        "magus": ["00"],   # OO → 00
    },
}
board_setup={2:board_setup_2p,3:board_setup_3p,4:board_setup_4p,5:board_setup_5p}
team_styles_5p = {
    "Blue": {
        "color": "#1f77b4",
        "circle_fill": "#ffffff",
    },
    "Green": {
        "color": "#2ca02c",
        "circle_fill": "#f8fff8",
    },
    "Orange": {
        "color": "#e67e22",
        "circle_fill": "#fffaf0",
    },
    "Pink": {
        "color": "#e83e8c",
        "circle_fill": "#fff7fb",
    },
    "Yellow": {
        "color": "#d4a017",
        "circle_fill": "#fffde7",
    },
}

In [10]:
import math
import pygame
HOME_SQUARES = {"1B", "3B", "5B", "7B", "00"}
TURN_ORDER_TABLE = {
    2: ["Orange", "Blue"],
    3: ["Orange", "Blue", "Green"],
    4: ["Blue", "Green", "Orange", "Pink"],
    5: ["Yellow", "Blue", "Green", "Orange", "Pink"],
}
# =========================================================
# 좌표 정규화
# =========================================================
def normalize_engine_square(square: str) -> str:
    square = square.strip().upper()
    if square == "OO":
        return "00"
    return square

def normalize_display_square(square: str) -> str:
    square = square.strip().upper()
    if square == "OO":
        return "00"
    return square


# =========================================================
# pygame 보드 렌더러
# =========================================================
class KaleidoBoardPygame:
    def __init__(
        self,
        board_size=920,
        margin=55,
        background_color=(255, 255, 255),
        board_fill=(242, 237, 184),
        palace_fill=(239, 231, 170),
        home_fill=(221, 213, 154),
        board_stroke=(34, 34, 34),
        stroke_width=2,
    ):
        self.board_size = board_size
        self.margin = margin
        self.background_color = background_color
        self.board_fill = board_fill
        self.palace_fill = palace_fill
        self.home_fill = home_fill
        self.board_stroke = board_stroke
        self.stroke_width = stroke_width

        self.cx = board_size / 2
        self.cy = board_size / 2

        # 링 경계
        self.r_boundaries = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]

        # 각 링의 분할 수
        self.ring_sectors = {
            1: 8,
            2: 16,
            3: 16,
            4: 24,
            5: 24,
        }

        self.max_r = self.r_boundaries[-1]
        self.scale = (board_size / 2 - margin) / self.max_r
        self.rotation_offset_deg = -22.5

        self.palace_cells = {
            (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7),
            (4, 0), (4, 1), (4, 2), (5, 0), (5, 2),
            (4, 6), (4, 7), (4, 8), (5, 6), (5, 8),
            (4, 12), (4, 13), (4, 14), (5, 12), (5, 14),
            (4, 18), (4, 19), (4, 20), (5, 18), (5, 20),
        }

        self.home_cells = {
            (5, 1),
            (5, 7),
            (5, 13),
            (5, 19),
        }

        self.letter_to_ring_local = {
            "G": (1, 0),
            "H": (2, 0), "F": (2, 1),
            "I": (3, 0), "E": (3, 1),
            "J": (4, 0), "A": (4, 1), "D": (4, 2),
            "K": (5, 0), "B": (5, 1), "C": (5, 2),
        }

        self.ring_to_letters = {
            1: ["G"],
            2: ["H", "F"],
            3: ["I", "E"],
            4: ["J", "A", "D"],
            5: ["K", "B", "C"],
        }

        self.labels = self.all_labels()

    # -------------------------
    # 기본 기하
    # -------------------------
    def scaled_r(self, r):
        return r * self.scale

    def polar_to_cart(self, r, deg):
        rad = math.radians(deg)
        rr = self.scaled_r(r)
        x = self.cx + rr * math.cos(rad)
        y = self.cy + rr * math.sin(rad)
        return x, y

    def label_to_ring_sector(self, label: str):
        label = label.strip().upper()
        if label in ("00", "OO"):
            return (0, 0)

        number = int(label[0])
        letter = label[1]

        ring, local = self.letter_to_ring_local[letter]
        group_size = len(self.ring_to_letters[ring])
        sector = (number - 1) * group_size + local
        return ring, sector

    def ring_sector_to_label(self, ring, sector):
        if ring == 0:
            return "00"

        letters = self.ring_to_letters[ring]
        group_size = len(letters)
        number = sector // group_size + 1
        local = sector % group_size
        return f"{number}{letters[local]}"

    def cell_center_polar(self, label: str):
        label = label.strip().upper()
        if label in ("00", "OO"):
            return 0.0, 0.0

        number = int(label[0])
        letter = label[1]

        ring, local = self.letter_to_ring_local[letter]
        group_size = len(self.ring_to_letters[ring])

        r_center = float(ring)
        slice_center_deg = -90 + self.rotation_offset_deg + (number - 1) * 45

        if group_size == 1:
            angle_offset = 22.5
        else:
            local_angle_step = 45 / group_size
            angle_offset = (local + 0.5) * local_angle_step

        angle_deg = slice_center_deg + angle_offset
        return r_center, angle_deg

    def cell_center_xy(self, label: str):
        r, deg = self.cell_center_polar(label)
        return self.polar_to_cart(r, deg)

    def all_labels(self):
        labels = ["00"]
        for n in range(1, 9):
            for ch in ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K"]:
                labels.append(f"{n}{ch}")
        return labels

    def cell_fill(self, ring_index=None, sector=None, is_center=False):
        if is_center:
            return self.home_fill
        if (ring_index, sector) in self.home_cells:
            return self.home_fill
        if (ring_index, sector) in self.palace_cells:
            return self.palace_fill
        return self.board_fill

    # -------------------------
    # 도형 점 집합
    # -------------------------
    def sector_polygon_points(self, ring_index, sector, points_per_arc=22):
        n = self.ring_sectors[ring_index]
        r_in = self.r_boundaries[ring_index - 1]
        r_out = self.r_boundaries[ring_index]
        angle_step = 360 / n

        a0 = -90 + self.rotation_offset_deg + sector * angle_step
        a1 = a0 + angle_step

        pts = []

        for i in range(points_per_arc + 1):
            t = i / points_per_arc
            a = a0 + (a1 - a0) * t
            pts.append(self.polar_to_cart(r_out, a))

        for i in range(points_per_arc, -1, -1):
            t = i / points_per_arc
            a = a0 + (a1 - a0) * t
            pts.append(self.polar_to_cart(r_in, a))

        return [(int(x), int(y)) for x, y in pts]

    def center_circle_radius(self):
        return int(self.scaled_r(self.r_boundaries[0]))

    # -------------------------
    # 렌더
    # -------------------------
    def draw_board(self, screen, highlighted=None):
        if highlighted is None:
            highlighted = {}

        screen.fill(self.background_color)

        center_label = "00"
        center_fill = highlighted.get(center_label, self.cell_fill(is_center=True))

        pygame.draw.circle(
            screen,
            center_fill,
            (int(self.cx), int(self.cy)),
            self.center_circle_radius()
        )
        pygame.draw.circle(
            screen,
            self.board_stroke,
            (int(self.cx), int(self.cy)),
            self.center_circle_radius(),
            self.stroke_width
        )

        for ring_index in range(1, 6):
            n = self.ring_sectors[ring_index]
            for sector in range(n):
                poly = self.sector_polygon_points(ring_index, sector)
                label = self.ring_sector_to_label(ring_index, sector)
                fill = highlighted.get(label, self.cell_fill(ring_index, sector))

                pygame.draw.polygon(screen, fill, poly)
                pygame.draw.polygon(screen, self.board_stroke, poly, self.stroke_width)

        pygame.draw.circle(
            screen,
            self.board_stroke,
            (int(self.cx), int(self.cy)),
            int(self.scaled_r(self.max_r)),
            self.stroke_width
        )

    def draw_labels(self, screen, font, color=(0, 0, 0)):
        for label in self.labels:
            x, y = self.cell_center_xy(label)
            text = font.render(label, True, color)
            rect = text.get_rect(center=(int(x), int(y)))
            screen.blit(text, rect)
    # -------------------------
    # SVG와 같은 심벌 체계용 기물 정규화
    # -------------------------
    def normalize_piece_code(self, piece: str):
        piece = piece.strip().upper()
        valid = {
            "X1", "X2", "X3", "X4",
            "Y1", "Y2", "Y3", "Y4",
            "Z1", "Z2", "Z3", "Z4",
        }
        if piece not in valid:
            raise ValueError(f"Invalid piece code: {piece}")
        if piece == "Z1":
            return "Y1"
        return piece

    # -------------------------
    # SVG와 동일한 내부 도형용 점 배치
    # -------------------------
    def _dot_positions(self, n, spacing):
        if n == 1:
            return [(0, 0)]
        if n == 2:
            return [(0, -spacing * 0.65), (0, spacing * 0.65)]
        if n == 3:
            return [(0, -spacing), (-spacing * 0.85, spacing * 0.55), (spacing * 0.85, spacing * 0.55)]
        if n == 4:
            return [(-spacing, -spacing), (spacing, -spacing), (-spacing, spacing), (spacing, spacing)]
        raise ValueError("dot count must be 1..4")

    # -------------------------
    # X 계열: 점들
    # -------------------------
    def _draw_x_piece_inner(self, screen, cx, cy, grade, color, dot_radius=5):
        spacing = 6.5
        for dx, dy in self._dot_positions(grade, spacing):
            pygame.draw.circle(
                screen,
                color,
                (int(round(cx + dx)), int(round(cy + dy))),
                int(round(dot_radius))
            )

    # -------------------------
    # Y 계열 basic: 세로선 여러 개
    # -------------------------
    def _draw_y_piece_inner_basic(self, screen, cx, cy, grade, color):
        h = 18
        gap = 5.5
        offsets = {
            1: [0],
            2: [-gap/2, gap/2],
            3: [-gap, 0, gap],
            4: [-1.5*gap, -0.5*gap, 0.5*gap, 1.5*gap],
        }[grade]

        for dx in offsets:
            x = cx + dx
            pygame.draw.line(
                screen,
                color,
                (int(round(x)), int(round(cy - h/2))),
                (int(round(x)), int(round(cy + h/2))),
                2
            )

    # -------------------------
    # Y 계열 readable: SVG readable과 동일
    # -------------------------
    def _draw_y_piece_inner_readable(self, screen, cx, cy, grade, color):
        if grade == 1:
            self._draw_y_piece_inner_basic(screen, cx, cy, 1, color)
            return

        if grade == 2:
            pygame.draw.line(
                screen, color,
                (int(round(cx - 7)), int(round(cy - 7))),
                (int(round(cx + 7)), int(round(cy + 7))),
                2
            )
            pygame.draw.line(
                screen, color,
                (int(round(cx - 7)), int(round(cy + 7))),
                (int(round(cx + 7)), int(round(cy - 7))),
                2
            )
            return

        if grade == 3:
            self._draw_y_piece_inner_basic(screen, cx, cy, 3, color)
            return

        if grade == 4:
            pygame.draw.line(
                screen, color,
                (int(round(cx - 7)), int(round(cy - 9))),
                (int(round(cx - 7)), int(round(cy + 9))),
                2
            )
            pygame.draw.line(
                screen, color,
                (int(round(cx + 7)), int(round(cy - 9))),
                (int(round(cx + 7)), int(round(cy + 9))),
                2
            )
            pygame.draw.line(
                screen, color,
                (int(round(cx - 9)), int(round(cy - 7))),
                (int(round(cx + 9)), int(round(cy - 7))),
                2
            )
            pygame.draw.line(
                screen, color,
                (int(round(cx - 9)), int(round(cy + 7))),
                (int(round(cx + 9)), int(round(cy + 7))),
                2
            )
            return

        raise ValueError("line grade must be 1..4")

    # -------------------------
    # Z 계열: SVG와 동일
    # -------------------------
    def _draw_z_piece_inner(self, screen, cx, cy, grade, color):
        if grade == 2:
            pygame.draw.line(
                screen, color,
                (int(round(cx + 0)), int(round(cy - 10))),
                (int(round(cx - 9)), int(round(cy + 7))),
                2
            )
            pygame.draw.line(
                screen, color,
                (int(round(cx + 0)), int(round(cy - 10))),
                (int(round(cx + 9)), int(round(cy + 7))),
                2
            )
            return

        if grade == 3:
            pts = [
                (int(round(cx + 0)), int(round(cy - 10))),
                (int(round(cx - 9)), int(round(cy + 7))),
                (int(round(cx + 9)), int(round(cy + 7))),
            ]
            pygame.draw.polygon(screen, color, pts, 2)
            return

        if grade == 4:
            lines = [
                ((0, -10), (0, 10)),
                ((-10, 0), (10, 0)),
                ((-7.2, -7.2), (7.2, 7.2)),
                ((-7.2, 7.2), (7.2, -7.2)),
            ]
            for (x1, y1), (x2, y2) in lines:
                pygame.draw.line(
                    screen, color,
                    (int(round(cx + x1)), int(round(cy + y1))),
                    (int(round(cx + x2)), int(round(cy + y2))),
                    2
                )
            return

        raise ValueError("Z piece grade must be 2..4")

    # -------------------------
    # piece_name -> SVG 코드 매핑
    # -------------------------
    def piece_name_to_code(self, piece_name: str):
        mapping = {
            "pawn": "X1",
            "horse": "X2",
            "tiger": "X3",
            "elephant": "X4",
            "raven": "Y1",
            "eagle": "Y2",
            "roc": "Y3",
            "fairy": "Z2",
            "wizard": "Z3",
            "magus": "Z4",
            "dragon": "Y4",   # 필요시 다른 코드로 바꿔도 됨
        }
        if piece_name not in mapping:
            raise ValueError(f"Unknown piece name: {piece_name}")
        return mapping[piece_name]

    # -------------------------
    # 내부 심벌 1개 그리기
    # mode: "simple" or "readable"
    # -------------------------
    def draw_piece_symbol(self, screen, cx, cy, piece_code, color=(20, 20, 20), mode="simple"):
        piece_code = self.normalize_piece_code(piece_code)
        kind = piece_code[0]
        grade = int(piece_code[1])

        if kind == "X":
            self._draw_x_piece_inner(screen, cx, cy, grade, color)

        elif kind == "Y":
            if mode == "simple":
                self._draw_y_piece_inner_basic(screen, cx, cy, grade, color)
            elif mode == "readable":
                self._draw_y_piece_inner_readable(screen, cx, cy, grade, color)
            else:
                raise ValueError("mode must be 'simple' or 'readable'")

        elif kind == "Z":
            self._draw_z_piece_inner(screen, cx, cy, grade, color)

        else:
            raise ValueError(f"Unknown piece kind: {kind}")
    def draw_piece_disc(
        self,
        screen,
        label,
        team_color,
        piece_name,
        radius=25,
        outline=(30, 30, 30),
        font=None,
        mode="simple",
    ):
        x, y = self.cell_center_xy(label)
        x = int(round(x))
        y = int(round(y))

        # 바탕 원
        pygame.draw.circle(screen, (255, 255, 255), (x, y), radius)
        pygame.draw.circle(screen, outline, (x, y), radius, 2)

        # 팀 색 내부 원
        pygame.draw.circle(screen, team_color, (x, y), max(8, radius // 2))

        # SVG와 동일한 심벌
        piece_code = self.piece_name_to_code(piece_name)
        self.draw_piece_symbol(
            screen,
            x,
            y,
            piece_code=piece_code,
            color=(20, 20, 20),
            mode=mode,
        )

        # 아래 약어 표시
        if font is not None:
            abbrev = {
                "pawn": "P",
                "horse": "H",
                "tiger": "T",
                "elephant": "E",
                "eagle": "EA",
                "roc": "R",
                "raven": "RA",
                "wizard": "W",
                "fairy": "F",
                "dragon": "D",
                "magus": "M",
            }.get(piece_name, piece_name[:1].upper())

            text = font.render(abbrev, True, (10, 10, 10))
            rect = text.get_rect(center=(x, y + radius + 12))
            screen.blit(text, rect)

    def draw_pieces(self, screen, board_map, team_colors, font=None, mode="simple"):
        for square, info in board_map.items():
            label = "00" if square == "OO" else square
            color = team_colors.get(info["color"], (0, 0, 0))
            self.draw_piece_disc(
                screen,
                label,
                color,
                info["piece"],
                font=font,
                mode=mode,
            )

    # -------------------------
    # 클릭 판정
    # -------------------------
    def nearest_label(self, pos, max_dist=34):
        px, py = pos
        best = None
        best_d2 = None

        for label in self.labels:
            x, y = self.cell_center_xy(label)
            d2 = (x - px) ** 2 + (y - py) ** 2
            if best_d2 is None or d2 < best_d2:
                best = label
                best_d2 = d2

        if best_d2 is None or best_d2 > max_dist ** 2:
            return None
        return best


# =========================================================
# 팀 색
# =========================================================
TEAM_COLORS = {
    "Blue":   (43, 89, 195),
    "Green":  (46, 139, 87),
    "Orange": (217, 119, 6),
    "Pink":   (217, 70, 160),
    "Yellow": (201, 161, 0),
}


# =========================================================
# 기존 legal_targets_for_piece(board_map, adj, color, piece, square)
# 를 pygame 앱에서 쓰기 위한 래퍼
# =========================================================
def make_legal_target_getter(adj, legal_targets_for_piece_func):
    def getter(board_map, color, piece_name, start_square):
        return legal_targets_for_piece_func(
            board_map, adj, color, piece_name, start_square
        )
    return getter


# =========================================================
# pygame 게임 앱
# =========================================================
class KaleidoGameApp:
    def __init__(self, board, board_map, legal_target_getter, piece_mode="simple",
    turn_order=None,
    home_squares={"1B", "3B", "5B", "7B", "00"},players=5):
        self.board = board
        self.board_map = {
            normalize_engine_square(k): dict(v) for k, v in board_map.items()
        }
        self.legal_target_getter = legal_target_getter
        self.piece_mode = piece_mode
        self.players=players
        
        if turn_order is not None:
            self.turn_order = turn_order
        else:
            self.turn_order = TURN_ORDER_TABLE[self.players]

        if home_squares is None:
            home_squares = HOME_SQUARES

        self.home_squares = {
            normalize_engine_square(square) for square in home_squares
        }

        self.alive_teams = set(self.turn_order)
        self.defeated_teams = set()
        self.current_turn_index = 0
        self.winner = None
        
        self.selected_square = None
        self.selected_info = None
        self.legal_targets = []

        pygame.init()
        self.status_h = 120
        self.screen = pygame.display.set_mode(
            (board.board_size, board.board_size + self.status_h)
        )
        pygame.display.set_caption("Kaleido Chess")
        self.clock = pygame.time.Clock()

        self.font = pygame.font.SysFont("malgungothic", 22)
        self.small_font = pygame.font.SysFont("malgungothic", 16)

        # 시작 직후 홈칸 점검
        self.update_defeated_teams()
        self.advance_turn_if_needed()        



    # -------------------------
    # 상태 갱신
    # -------------------------
    def clear_selection(self):
        self.selected_square = None
        self.selected_info = None
        self.legal_targets = []

    def select_piece(self, square):
        info = self.board_map[square]
        self.selected_square = square
        self.selected_info = info
        self.legal_targets = [
            normalize_engine_square(sq)
            for sq in self.legal_target_getter(
                self.board_map,
                info["color"],
                info["piece"],
                square,
            )
        ]

        print(
            f"selected: {normalize_display_square(square)} | "
            f"team={info['color']} | piece={info['piece']}"
        )

    def move_selected_to(self, target_square):
        if self.selected_square is None or self.selected_info is None:
            return

        if self.winner is not None:
            return

        src = self.selected_square
        dst = target_square
        moving_info = dict(self.selected_info)

        overwritten = self.board_map.get(dst)

        # 자기편이든 남의 편이든 덮어쓰기 가능
        if dst in self.board_map:
            del self.board_map[dst]

        del self.board_map[src]
        self.board_map[dst] = moving_info

        print(
            f"moved: {normalize_display_square(src)} -> {normalize_display_square(dst)} | "
            f"team={moving_info['color']} | piece={moving_info['piece']}"
        )

        if overwritten is not None:
            print(
                f"overwritten: {normalize_display_square(dst)} had "
                f"{overwritten['color']} {overwritten['piece']}"
            )

        self.clear_selection()

        # 홈칸 비었는지 검사 -> 패배 처리
        self.update_defeated_teams()

        # 승자 없으면 다음 턴
        if self.winner is None:
            self.advance_turn()
            self.advance_turn_if_needed()

    # -------------------------
    # 입력 처리
    # -------------------------
    def handle_click(self, pos):
        if self.winner is not None:
            return

        if pos[1] > self.board.board_size:
            return

        label = self.board.nearest_label(pos)
        if label is None:
            return

        engine_sq = normalize_engine_square(label)

        # 이미 선택된 상태에서 이동 가능 칸 클릭 -> 실제 이동
        if self.selected_square is not None and engine_sq in self.legal_targets:
            self.move_selected_to(engine_sq)
            return

        # 오직 기물이 있는 칸만 새로 선택 가능
        if engine_sq in self.board_map:
            info = self.board_map[engine_sq]

            # 현재 턴 팀만 선택 가능
            if info["color"] != self.current_team():
                return

            self.select_piece(engine_sq)
            return

        return
    # -------------------------
    # 렌더용 하이라이트
    # -------------------------
    def build_highlight_map(self):
        highlight = {}

        # 공용 홈칸 5개 연한 강조
        for sq in self.home_squares:
            show_sq = normalize_display_square(sq)
            highlight[show_sq] = (255, 245, 170)

        # 이동 가능 칸
        for sq in self.legal_targets:
            show_sq = normalize_display_square(sq)
            highlight[show_sq] = (102, 204, 255)

        # 선택 칸
        if self.selected_square is not None:
            show_sq = normalize_display_square(self.selected_square)
            highlight[show_sq] = (255, 120, 120)

        return highlight

    # -------------------------
    # 하단 상태바
    # -------------------------
    def draw_status_bar(self):
        y0 = self.board.board_size
        pygame.draw.rect(
            self.screen,
            (245, 245, 245),
            (0, y0, self.board.board_size, self.status_h)
        )
        pygame.draw.line(
            self.screen,
            (180, 180, 180),
            (0, y0),
            (self.board.board_size, y0),
            1
        )

        if self.winner is not None:
            text = f"게임 종료 | 승자: {self.winner}"
        elif self.selected_info is None:
            text = (
                f"현재 턴: {self.current_team()} | "
                f"홈칸에 1개라도 자기 기물이 있어야 생존 | "
                f"현재 턴의 기물만 클릭 가능"
            )
        else:
            text = (
                f"현재 턴: {self.current_team()} | "
                f"선택: {normalize_display_square(self.selected_square)} | "
                f"팀: {self.selected_info['color']} | "
                f"기물: {self.selected_info['piece']} | "
                f"가능 칸 수: {len(self.legal_targets)}"
            )

        surf = self.font.render(text, True, (20, 20, 20))
        self.screen.blit(surf, (14, y0 + 12))

        defeated_text = "탈락: " + (", ".join(t for t in self.turn_order if t in self.defeated_teams) or "없음")
        surf_def = self.small_font.render(defeated_text, True, (80, 80, 80))
        self.screen.blit(surf_def, (14, y0 + 45))

        if self.legal_targets:
            preview = ", ".join(
                normalize_display_square(sq) for sq in self.legal_targets[:14]
            )
            if len(self.legal_targets) > 14:
                preview += " ..."
            surf2 = self.small_font.render(preview, True, (65, 65, 65))
            self.screen.blit(surf2, (14, y0 + 75))

    # -------------------------
    # 전체 그리기
    # -------------------------
    def draw(self):
        highlight = self.build_highlight_map()

        board_surface = pygame.Surface((self.board.board_size, self.board.board_size))
        self.board.draw_board(board_surface, highlighted=highlight)
        self.board.draw_pieces(
            board_surface,
            self.board_map,
            TEAM_COLORS,
            font=self.small_font,
            mode=self.piece_mode,
        )
        # 라벨이 필요하면 아래 줄 주석 해제
        # self.board.draw_labels(board_surface, self.small_font)

        self.screen.blit(board_surface, (0, 0))
        self.draw_status_bar()
        pygame.display.flip()

    def current_team(self):
        if self.winner is not None:
            return None
        if not self.turn_order:
            return None
        return self.turn_order[self.current_turn_index]

    def team_has_home_piece(self, team):
        for sq in self.home_squares:
            if sq in self.board_map and self.board_map[sq]["color"] == team:
                return True
        return False

    def update_defeated_teams(self):
        changed = False

        for team in self.turn_order:
            if team in self.defeated_teams:
                continue

            if not self.team_has_home_piece(team):
                self.defeated_teams.add(team)
                if team in self.alive_teams:
                    self.alive_teams.remove(team)
                changed = True
                print(f"defeated: {team} (home square empty)")

        alive_list = [t for t in self.turn_order if t in self.alive_teams]
        if len(alive_list) == 1:
            self.winner = alive_list[0]
            print(f"winner: {self.winner}")

        return changed

    def advance_turn(self):
        if self.winner is not None:
            return

        n = len(self.turn_order)
        if n == 0:
            return

        for _ in range(n):
            self.current_turn_index = (self.current_turn_index + 1) % n
            if self.turn_order[self.current_turn_index] in self.alive_teams:
                return

    def advance_turn_if_needed(self):
        if self.winner is not None:
            return

        n = len(self.turn_order)
        if n == 0:
            return

        for _ in range(n):
            team = self.turn_order[self.current_turn_index]
            if team in self.alive_teams:
                return
            self.current_turn_index = (self.current_turn_index + 1) % n

    
    # -------------------------
    # 실행
    # -------------------------
    def run(self):
        running = True
        while running:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
                elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                    self.handle_click(event.pos)

            self.draw()
            self.clock.tick(60)

        pygame.quit()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.4)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [11]:
board = KaleidoBoardPygame(board_size=920)
legal_getter = make_legal_target_getter(adj, legal_targets_for_piece)

app = KaleidoGameApp(
    board=board,
    board_map=board_map,
    legal_target_getter=legal_getter,
    piece_mode="simple",
)
app.run()

In [12]:
board = KaleidoBoardPygame(board_size=920)
legal_getter = make_legal_target_getter(adj, legal_targets_for_piece)
players=2
app = KaleidoGameApp(
    board=board,
    board_map=team_based_to_board_map(board_setup[players]),
    legal_target_getter=legal_getter,
    piece_mode="readable",
    turn_order=TURN_ORDER_TABLE[players],
    home_squares=HOME_SQUARES,players=players
)
app.run()

In [13]:
legal_targets_for_piece

<function __main__.legal_targets_for_piece(board_map, adj, color, piece_name, start_square)>

In [14]:
#        "fairy":   ["3A","3B","3C","3D","3E","3F","3G","3H","3I","3J","3K","00"], #debug